# Semana 28 — Difference-in-Differences (DiD)
**Fase 6: Causal Inference e Experimentation** · Gabarito (notebook resolvido)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/applied-ai-architecture/blob/main/06-Phase-6-Causal-Inference/notebooks/semana_28_gabarito.ipynb)

Leitura de conceito desta semana: [Semana-28-Conceitos.md](https://github.com/NetoRibeiro/applied-ai-architecture/blob/main/06-Phase-6-Causal-Inference/docs/Semana-28-Conceitos.md)

Use este gabarito **depois** de tentar o notebook de exercícios. Compare a sua solução, não copie.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.formula.api as smf
rng = np.random.default_rng(0)

## Passo 1 — O cenário
Um novo sistema de recomendação por IA é lançado na semana 20 **apenas em São Paulo**. Temos a receita semanal por loja em três cidades por 40 semanas. Efeito verdadeiro em SP: **+8,0** por loja/semana.

In [ ]:
EFEITO, T0, SEMANAS = 8.0, 20, 40
def gerar(spillover_rj=0.0, tendencia_extra_sp=0.0, seed=0):
    r = np.random.default_rng(seed); linhas = []
    niveis = {"SP": 120, "RJ": 95, "BH": 80}
    for cidade, nivel in niveis.items():
        for loja in range(60):
            base = nivel + r.normal(0, 6)
            for s in range(SEMANAS):
                y = base + 0.5 * s + 4 * np.sin(2 * np.pi * s / 13) + r.normal(0, 3)       # tendência comum + sazonalidade
                if cidade == "SP": y += tendencia_extra_sp * s + (EFEITO if s >= T0 else 0)
                if cidade == "RJ" and s >= T0: y += spillover_rj
                linhas.append((f"{cidade}-{loja}", cidade, s, y))
    return pd.DataFrame(linhas, columns=["loja", "cidade", "semana", "receita"])

df = gerar()
df.groupby(["semana", "cidade"])["receita"].mean().unstack().plot(figsize=(9, 3.5)); plt.axvline(T0, ls="--", c="gray"); plt.ylabel("receita média por loja"); plt.show()

## Passo 2 — A comparação ingênua
Duas respostas erradas e populares: (a) SP depois vs. SP antes; (b) SP vs. RJ no período depois.

In [ ]:
d = df[df.cidade.isin(["SP", "RJ"])].assign(tratado=lambda x: (x.cidade == "SP").astype(int), pos=lambda x: (x.semana >= T0).astype(int))
m = d.groupby(["tratado", "pos"])["receita"].mean()
antes_depois = m[1, 1] - m[1, 0]
sp_vs_rj = m[1, 1] - m[0, 1]
print(f"antes/depois em SP: {antes_depois:.1f} | SP − RJ depois: {sp_vs_rj:.1f} | verdade: {EFEITO}")

Por que cada uma erra?

## Passo 3 — DiD pela tabela 2×2

In [ ]:
did = (m[1, 1] - m[1, 0]) - (m[0, 1] - m[0, 0])
print(m.unstack().round(1)); print(f"\nDiD = {did:.2f} | verdade: {EFEITO}")

## Passo 4 — DiD por regressão, com erro-padrão por loja
Ajuste `receita ~ tratado * pos` com cluster-robust standard errors por loja. O coeficiente `tratado:pos` deve coincidir com a conta do Passo 3.

In [ ]:
mod = smf.ols("receita ~ tratado * pos", d).fit(cov_type="cluster", cov_kwds={"groups": d["loja"]})
b = mod.params["tratado:pos"]; ic = mod.conf_int().loc["tratado:pos"]
print(f"DiD = {b:.2f} | IC95% [{ic[0]:.2f}, {ic[1]:.2f}] | p = {mod.pvalues['tratado:pos']:.2g}")
assert abs(b - did) < 1e-6

## Passo 5 — Parallel trends: placebo test
Use **só o período anterior** (semanas 0 a 19) e finja que a intervenção ocorreu na semana 10. O efeito estimado deve ser próximo de zero.

In [ ]:
def placebo(dados, semana_falsa=10):
    pre = dados[dados.semana < T0].assign(pos=lambda x: (x.semana >= semana_falsa).astype(int))
    mp = smf.ols("receita ~ tratado * pos", pre).fit(cov_type="cluster", cov_kwds={"groups": pre["loja"]})
    return mp.params["tratado:pos"], mp.pvalues["tratado:pos"]

print("placebo: efeito = %.2f | p = %.2f" % placebo(d))

## Passo 6 — Event study
Estime um efeito por semana em relação à semana anterior ao lançamento (referência: semana 19). Coeficientes antes do lançamento em torno de zero sustentam a hipótese; depois, mostram a dinâmica do efeito.

In [ ]:
d["rel"] = d["semana"] - T0
mod_ev = smf.ols("receita ~ C(rel, Treatment(reference=-1)) * tratado", d).fit(cov_type="cluster", cov_kwds={"groups": d["loja"]})
coefs = {int(k.split("[T.")[1].split("]")[0]): v for k, v in mod_ev.params.items() if k.endswith(":tratado")}
ics = {int(k.split("[T.")[1].split("]")[0]): tuple(v) for k, v in mod_ev.conf_int().iterrows() if k.endswith(":tratado")}
xs = sorted(coefs); plt.figure(figsize=(9, 3.5))
plt.errorbar(xs, [coefs[x] for x in xs], yerr=[[coefs[x] - ics[x][0] for x in xs], [ics[x][1] - coefs[x] for x in xs]], fmt="o", ms=3)
plt.axhline(0, c="k", lw=.8); plt.axvline(-0.5, ls="--", c="gray"); plt.axhline(EFEITO, ls=":", c="g"); plt.xlabel("semanas desde o lançamento"); plt.ylabel("efeito estimado"); plt.show()

## Passo 7 — Quando parallel trends não vale
SP já crescia 0,3 por semana a mais que o RJ. Refaça o DiD e o placebo.

In [ ]:
df2 = gerar(tendencia_extra_sp=0.3)
d2 = df2[df2.cidade.isin(["SP", "RJ"])].assign(tratado=lambda x: (x.cidade == "SP").astype(int), pos=lambda x: (x.semana >= T0).astype(int))
did2 = smf.ols("receita ~ tratado * pos", d2).fit().params["tratado:pos"]
pl2 = placebo(d2)
print(f"DiD = {did2:.2f} (verdade {EFEITO}) | placebo: efeito = {pl2[0]:.2f}, p = {pl2[1]:.3g}")

O placebo acusou o problema? Sem ele, você teria reportado um efeito inflado.

## Passo 8 — Spillover: o controle contaminado
A campanha da IA em SP viralizou e aumentou as vendas do RJ em +3,0 depois do lançamento. Estime o DiD com RJ como controle.

In [ ]:
df3 = gerar(spillover_rj=3.0)
def did_com(dados, controle):
    x = dados[dados.cidade.isin(["SP", controle])].assign(tratado=lambda z: (z.cidade == "SP").astype(int), pos=lambda z: (z.semana >= T0).astype(int))
    mm = smf.ols("receita ~ tratado * pos", x).fit(cov_type="cluster", cov_kwds={"groups": x["loja"]})
    return mm.params["tratado:pos"], tuple(mm.conf_int().loc["tratado:pos"])

e_rj, ic_rj = did_com(df3, "RJ"); print(f"controle RJ (contaminado): {e_rj:.2f} IC [{ic_rj[0]:.2f}, {ic_rj[1]:.2f}] | verdade: {EFEITO}")

## Passo 9 — Corrigindo com um controle não exposto
Belo Horizonte não foi alcançada pela campanha. Use BH como controle. Depois, use o próprio RJ como "tratado pelo spillover" contra BH para **medir** o vazamento.

In [ ]:
e_bh, ic_bh = did_com(df3, "BH"); print(f"controle BH (limpo): {e_bh:.2f} IC [{ic_bh[0]:.2f}, {ic_bh[1]:.2f}]")
x = df3[df3.cidade.isin(["RJ", "BH"])].assign(tratado=lambda z: (z.cidade == "RJ").astype(int), pos=lambda z: (z.semana >= T0).astype(int))
vazamento = smf.ols("receita ~ tratado * pos", x).fit().params["tratado:pos"]
print(f"spillover estimado no RJ: {vazamento:.2f} (verdade: 3.0)")

## Teste Prático de Entrevista — Estimador Difference-in-Differences (DiD 2x2)
Um entrevistador pede: *"Implemente `did_2x2(treat_pre, treat_post, ctrl_pre, ctrl_post)` retornando a estimativa DiD `(mean(treat_post) - mean(treat_pre)) - (mean(ctrl_post) - mean(ctrl_pre))` e demonstrando por que comparar apenas `treat_post - treat_pre` superestimaria o efeito quando todo o mercado cresce."* 

In [ ]:
def did_2x2(treat_pre, treat_post, ctrl_pre, ctrl_post):
    """Calcula o efeito causal Difference-in-Differences 2x2."""
    mean = lambda xs: sum(xs) / len(xs)
    diff_treat = mean(treat_post) - mean(treat_pre)
    diff_ctrl = mean(ctrl_post) - mean(ctrl_pre)
    return diff_treat - diff_ctrl


eff = did_2x2([10, 12], [15, 17], [8, 10], [11, 13])
assert abs(eff - 2.0) < 1e-9, f"Esperado 2.0, veio {eff}"
print(f"OK! Efeito DiD={eff:.2f}")

## Fechamento
Escreva o parágrafo para o diretor. Inclua: a estimativa com intervalo, por que RJ não serve como controle, por que BH serve, o que o placebo mostrou e o fato de que o impacto total da campanha no negócio inclui o ganho indireto das lojas do RJ.